In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from src.shared_preprocessing import RANDOM_STATE, make_cv

SEED = RANDOM_STATE

models = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, random_state=SEED),
        {"model__C": [0.01, 0.1, 1, 10], "model__class_weight": [None, "balanced"]},
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=SEED),
        {
            "model__max_depth": [None, 5, 10, 20],
            "model__min_samples_leaf": [1, 5, 20],
            "model__class_weight": [None, "balanced"],
        },
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=SEED, n_jobs=1),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_leaf": [1, 5],
            "model__class_weight": [None, "balanced"],
        },
    ),
}

In [2]:
def run_base_models(preprocessor, X_train, X_test, y_train, y_test, *, scoring):
    """Compare baselines with tuned models using shared folds and a held-out test set."""
    cv = make_cv()
    summary = []
    test_scores = []
    confusion_matrices = {}
    tuned_models = {}

    for name, (estimator, grid) in models.items():
        pipeline = Pipeline([
            ("preprocess", clone(preprocessor)),
            ("model", clone(estimator)),
        ])
        baseline_cv = cross_val_score(
            pipeline, X_train, y_train, scoring=scoring, cv=cv, error_score="raise",
        ).mean()
        search = GridSearchCV(
            pipeline, grid, scoring=scoring, cv=cv, n_jobs=-1, error_score="raise",
        )
        search.fit(X_train, y_train)
        tuned_models[name] = search.best_estimator_
        summary.append({
            "model": name,
            "baseline_cv_mean": baseline_cv,
            "tuned_cv_mean": search.best_score_,
            "cv_change": search.best_score_ - baseline_cv,
            "best_parameters": search.best_params_,
        })

        fitted_model = search.best_estimator_
        predictions = fitted_model.predict(X_test)
        probabilities = fitted_model.predict_proba(X_test)[:, 1]
        test_scores.append({
            "model": name,
            "stage": "tuned",
            "accuracy": accuracy_score(y_test, predictions),
            "precision": precision_score(y_test, predictions, zero_division=0),
            "recall": recall_score(y_test, predictions, zero_division=0),
            "f1": f1_score(y_test, predictions, zero_division=0),
            "roc_auc": roc_auc_score(y_test, probabilities),
        })
        confusion_matrices[(name, "tuned")] = confusion_matrix(
            y_test, predictions, labels=[0, 1],
        )


    return pd.DataFrame(summary), pd.DataFrame(test_scores), confusion_matrices, tuned_models

In [3]:
from src.shared_preprocessing import PRIMARY_SCORING, build_preprocessor, get_split

X_train, X_test, y_train, y_test = get_split()
preprocessor = build_preprocessor(X_train)

tuning_summary, test_results, matrices, tuned_models = run_base_models(
    preprocessor, X_train, X_test, y_train, y_test, scoring=PRIMARY_SCORING
)
display(tuning_summary, test_results)


,model,baseline_cv_mean,tuned_cv_mean,cv_change,best_parameters
0,Logistic Regression,0.274744,0.372762,0.098018,"{'model__C': 10, 'model__class_weight': 'balan..."
1,Decision Tree,0.311191,0.413165,0.101974,"{'model__class_weight': 'balanced', 'model__ma..."
2,Random Forest,0.334746,0.471861,0.137115,"{'model__class_weight': 'balanced', 'model__ma..."


,model,stage,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,tuned,0.755612,0.266990,0.623819,0.373938,0.772318
1,Decision Tree,tuned,0.814774,0.335642,0.595463,0.429302,0.777822
2,Random Forest,tuned,0.866969,0.443842,0.541588,0.487867,0.804710
